In [1]:
import sys
sys.path.append('..')
  
from src.rag.retrieval import retrieve
from src.rag.reranker import simple_rerank


In [2]:

queries = [
    "recomendação de investimento para perfil conservador",
    "produtos de alto risco para cliente agressivo",
    "comunicação adequada com cliente sobre risco"
]


In [3]:

print("=== ANTES DO RE-RANKING ===\n")
for query in queries:
    print(f"Query: {query}")
    results = retrieve(query, n_results=3)
    for i, r in enumerate(results):
        print(f"  {i+1}. [{r['source']}] {r['content'][:80]}...")
    print()


=== ANTES DO RE-RANKING ===

Query: recomendação de investimento para perfil conservador
caminho do banco: c:\Users\GQ427MZ\OneDrive - EY\Desktop\projeto ai engineer\development-program-ai_engineer\projects\project\notebooks\..\data\chroma_db
Banco existe: True
total de chunks no banco: 22
  1. [analise_de_perfil_do_investidor.txt] va em seus objetivos ou situação financeira.
...
  2. [manual_comunicacao_cliente_v1.0.txt] o de gírias ou linguagem excessivamente informal que possa ser mal interpretada....
  3. [politica_adequacao_investimento_v1.2.txt] recomendação de produtos de Risco Alto deve ser acompanhada de um "Termo de Ciên...

Query: produtos de alto risco para cliente agressivo
caminho do banco: c:\Users\GQ427MZ\OneDrive - EY\Desktop\projeto ai engineer\development-program-ai_engineer\projects\project\notebooks\..\data\chroma_db
Banco existe: True
total de chunks no banco: 22
  1. [analise_de_perfil_do_investidor.txt] va em seus objetivos ou situação financeira.
...
  2. [poli

In [4]:

print("=== DEPOIS DO RE-RANKING ===\n")
for query in queries:
    print(f"Query: {query}")
    results = retrieve(query, n_results=3)
    reranked = simple_rerank(query, results)
    for i, r in enumerate(reranked):
        print(f"  {i+1}. [{r['source']}] score={r['rerank_score']} | {r['content'][:80]}...")
    print()

=== DEPOIS DO RE-RANKING ===

Query: recomendação de investimento para perfil conservador
caminho do banco: c:\Users\GQ427MZ\OneDrive - EY\Desktop\projeto ai engineer\development-program-ai_engineer\projects\project\notebooks\..\data\chroma_db
Banco existe: True
total de chunks no banco: 22
  1. [manual_comunicacao_cliente_v1.0.txt] score=2 | o de gírias ou linguagem excessivamente informal que possa ser mal interpretada....
  2. [politica_adequacao_investimento_v1.2.txt] score=2 | recomendação de produtos de Risco Alto deve ser acompanhada de um "Termo de Ciên...
  3. [analise_de_perfil_do_investidor.txt] score=0 | va em seus objetivos ou situação financeira.
...

Query: produtos de alto risco para cliente agressivo
caminho do banco: c:\Users\GQ427MZ\OneDrive - EY\Desktop\projeto ai engineer\development-program-ai_engineer\projects\project\notebooks\..\data\chroma_db
Banco existe: True
total de chunks no banco: 22
  1. [politica_adequacao_investimento_v1.2.txt] score=4 | recomendação 

In [6]:
ground_truth = {
    "recomendação de investimento para perfil conservador": "politica_adequacao_investimento_v1.2.txt",
    "produtos de alto risco para cliente agressivo": "politica_investimento_agressivo_v1.0.txt",
    "comunicação adequada com cliente sobre risco": "manual_comunicacao_cliente_v1.0.txt"
}

def reciprocal_rank(results, correct_doc):
    for i, r in enumerate(results):
        if r['source'] == correct_doc:
            return 1 / (i + 1)
    return 0

mrr_before = 0
mrr_after = 0

for query, correct_doc in ground_truth.items():
    results = retrieve(query, n_results=3)
    reranked = simple_rerank(query, list(results))
    
    rr_before = reciprocal_rank(results, correct_doc)
    rr_after = reciprocal_rank(reranked, correct_doc)
    
    mrr_before += rr_before
    mrr_after += rr_after
    
    print(f"Query: {query}")
    print(f"  RR antes: {rr_before:.2f} | RR depois: {rr_after:.2f}")

mrr_before /= len(ground_truth)
mrr_after /= len(ground_truth)

print(f"\n=== RESULTADO FINAL ===")
print(f"MRR antes do re-ranking:  {mrr_before:.3f}")
print(f"MRR depois do re-ranking: {mrr_after:.3f}")
print(f"Melhoria: {(mrr_after - mrr_before) / mrr_before * 100:.1f}%")

caminho do banco: c:\Users\GQ427MZ\OneDrive - EY\Desktop\projeto ai engineer\development-program-ai_engineer\projects\project\notebooks\..\data\chroma_db
Banco existe: True
total de chunks no banco: 22
Query: recomendação de investimento para perfil conservador
  RR antes: 0.33 | RR depois: 0.50
caminho do banco: c:\Users\GQ427MZ\OneDrive - EY\Desktop\projeto ai engineer\development-program-ai_engineer\projects\project\notebooks\..\data\chroma_db
Banco existe: True
total de chunks no banco: 22
Query: produtos de alto risco para cliente agressivo
  RR antes: 0.00 | RR depois: 0.00
caminho do banco: c:\Users\GQ427MZ\OneDrive - EY\Desktop\projeto ai engineer\development-program-ai_engineer\projects\project\notebooks\..\data\chroma_db
Banco existe: True
total de chunks no banco: 22
Query: comunicação adequada com cliente sobre risco
  RR antes: 0.50 | RR depois: 1.00

=== RESULTADO FINAL ===
MRR antes do re-ranking:  0.278
MRR depois do re-ranking: 0.500
Melhoria: 80.0%
